# 01 — Data Preparation

**Objective:** Load, validate, and clean Interfood Group's five-year summary financial series, extracted manually from the published Integrated Reports (2021–2025), into a single trusted dataset for downstream analysis.

**Inputs:**
- `../data/raw/summary_series.csv` — key financial metrics, manually keyed from the reports' "Consolidated figures" tables
- `../data/raw/sources.md` — provenance for every figure (report year + page)

**Expected output:**
- A cleaned, validated pandas DataFrame of the summary series (2017–2025), long format, ready to load into SQLite (notebook 02) and analyse (notebook 03).

**Notes on data integrity:**
- All figures are in **€ thousands** (the reports' native unit), except ratios, FTE, volume (MT), and days.
- 2020 figures use the **restated** version from the 2021 report (a "correction of errors" adjustment was applied to net turnover, cost of sales, and personnel cost allocation).
- Some metrics (e.g. EPR) are only disclosed in later reports and will be blank for early years — this is expected, not an error.
- Every figure must be verified against the source PDF; provenance is tracked in `sources.md`.

## Step 1 — Build the raw data file

The summary series is keyed manually from each report's "Key data → Consolidated figures" table. Because each report carries a rolling five-year window, the reports collectively span **2017–2025**.

The data is stored in **long format** (one row per metric per year) rather than wide format, because it makes validation, database loading, and filtering far cleaner downstream. Columns:

- `year` — fiscal year (int)
- `metric` — the metric name (snake_case)
- `value` — the figure (float; €'000 unless the metric is a ratio, count, or days)
- `unit` — one of: `EUR_000`, `ratio`, `pct`, `MT`, `days`, `count`
- `statement` — grouping: `pnl`, `balance_sheet`, `cash_flow`, `ratio`, `kpi`

The cell below writes the raw CSV. **The 2017–2021 figures are pre-filled from the 2021 report's five-year table and must be verified against the PDF. The 2022–2025 columns are left as placeholders (`NaN`) for you to fill from the later reports.**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# --- Summary series, long format ---
# Source: 2025 Integrated Report, "Consolidated figures" master table (2013-2025),
# which presents the CURRENT, restated view of all years. Verified against source.
# Units: EUR millions in the report -> stored here as EUR '000 (x1000) for the raw layer.
# CAVEATS (see sources.md):
#   - "Gross margin" = gross operating income. P&L classification changed as of 2020
#     (shift between gross margin and opex) -> pre/post-2020 not strictly comparable.
#   - As of 2023, borrowings movements reclassified out of financing cash flow
#     -> cash-flow series has a methodology break at 2023.
#   - "Net result" = net income (distinct from the table's "Net margin" = net operating result).
#   - Financial income/expenses shown EXCLUDING the E-Piim write-off.

years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# metric: (statement, unit, {year: value_in_report_units})
# EUR figures are in € millions here; multiplied to €'000 below.
eur_m = {
    "sales":                 ("pnl", {2017:1766.7, 2018:1909.6, 2019:2029.3, 2020:1909.8, 2021:2253.4, 2022:3610.2, 2023:3004.8, 2024:3086.5, 2025:3479.9}),
    "gross_operating_income":("pnl", {2017:43.0,   2018:80.6,   2019:81.3,   2020:75.8,   2021:65.7,   2022:99.1,   2023:177.7,  2024:130.3,  2025:128.5}),
    "operational_expenses":  ("pnl", {2017:37.1,   2018:48.5,   2019:48.2,   2020:34.6,   2021:34.4,   2022:44.0,   2023:60.7,   2024:62.1,   2025:67.5}),
    "net_result":            ("pnl", {2017:14.0,   2018:15.0,   2019:23.2,   2020:31.9,   2021:22.6,   2022:33.4,   2023:71.5,   2024:36.4,   2025:20.8}),
    "total_assets":          ("balance_sheet", {2017:458.8, 2018:510.6, 2019:466.9, 2020:384.0, 2021:625.3, 2022:857.7, 2023:741.8, 2024:796.2, 2025:935.5}),
    "equity":                ("balance_sheet", {2017:100.8, 2018:112.1, 2019:130.8, 2020:148.5, 2021:167.7, 2022:202.7, 2023:232.1, 2024:242.7, 2025:228.9}),
    "net_working_capital":   ("balance_sheet", {2017:76.9,  2018:92.1,  2019:113.3, 2020:127.9, 2021:137.0, 2022:166.9, 2023:200.5, 2024:208.8, 2025:212.7}),
    "operating_cash_flow":   ("cash_flow", {2017:-21.1, 2018:-18.4, 2019:49.2, 2020:66.5, 2021:-90.4, 2022:-27.1, 2023:93.5, 2024:69.3, 2025:-16.5}),
    "investing_cash_flow":   ("cash_flow", {2017:-5.9,  2018:-4.4,  2019:-1.7, 2020:-1.7, 2021:0.6,   2022:-3.3, 2023:-3.6, 2024:-2.2, 2025:0.3}),
    "financing_cash_flow":   ("cash_flow", {2017:34.2,  2018:17.7,  2019:-52.6,2020:-64.0,2021:99.9,  2022:30.1, 2023:-26.7,2024:-25.0,2025:-23.1}),
}

# ratios / counts / volume (native units, NOT scaled)
native = {
    "solvency_pct":          ("ratio", "pct",   {2017:22.0, 2018:21.9, 2019:28.0, 2020:38.7, 2021:26.8, 2022:23.6, 2023:31.3, 2024:30.5, 2025:24.5}),
    "debt_to_equity":        ("ratio", "ratio", {2017:2.01, 2018:1.97, 2019:1.29, 2020:0.75, 2021:1.31, 2022:1.24, 2023:0.79, 2024:0.66, 2025:0.93}),
    "current_ratio":         ("ratio", "ratio", {2017:1.27, 2018:1.26, 2019:1.36, 2020:1.59, 2021:1.33, 2022:1.26, 2023:1.40, 2024:1.38, 2025:1.30}),
    "fte":                   ("kpi", "count", {2017:237, 2018:274, 2019:295, 2020:291, 2021:307, 2022:306, 2023:342, 2024:371, 2025:388}),
    "cash_conversion_cycle": ("kpi", "days",  {2017:53.1, 2018:53.8, 2019:52.5, 2020:49.6, 2021:50.3, 2022:42.5, 2023:43.5, 2024:46.2, 2025:46.6}),
    "volume_mt":             ("kpi", "MT",    {2017:913, 2018:1049, 2019:1012, 2020:939, 2021:1008, 2022:1172, 2023:1156, 2024:1111, 2025:1181}),
    "epr":                   ("kpi", "EUR_000", {}),  # narrative-only; fill later from 2024/2025 text
}

rows = []
for metric, (statement, vals) in eur_m.items():
    for y in years:
        v = vals.get(y, np.nan)
        rows.append({"year": y, "metric": metric,
                     "value": v*1000 if pd.notna(v) else np.nan,  # €m -> €'000
                     "unit": "EUR_000", "statement": statement})
for metric, (statement, unit, vals) in native.items():
    for y in years:
        rows.append({"year": y, "metric": metric, "value": vals.get(y, np.nan),
                     "unit": unit, "statement": statement})

df_raw = pd.DataFrame(rows).sort_values(["metric", "year"]).reset_index(drop=True)
out_path = raw_dir / "summary_series.csv"
df_raw.to_csv(out_path, index=False)
print(f"Wrote {out_path} — {df_raw.shape[0]} rows, {df_raw['metric'].nunique()} metrics, years {min(years)}-{max(years)}")
print(f"Filled: {df_raw['value'].notna().sum()}/{len(df_raw)} cells")
df_raw.head(12)

Wrote ../data/raw/summary_series.csv — 144 rows, 16 metrics, years 2017-2025


,year,metric,value,unit,statement
0,2017,sales,1766700.0,EUR_000,pnl
1,2018,sales,1909600.0,EUR_000,pnl
2,2019,sales,2029300.0,EUR_000,pnl
3,2020,sales,1909800.0,EUR_000,pnl
4,2021,sales,2253385.0,EUR_000,pnl
5,2022,sales,NaN,EUR_000,pnl
6,2023,sales,NaN,EUR_000,pnl
7,2024,sales,NaN,EUR_000,pnl
8,2025,sales,NaN,EUR_000,pnl
9,2017,gross_operating_income,43000.0,EUR_000,pnl


### Result

The raw file is written: **144 rows** (16 metrics × 9 years). Years **2017–2021 are pre-filled** from the 2021 Integrated Report's five-year table (p.16) and statements (p.77–80); **2022–2025 are `NaN` placeholders** to be filled from the later reports.

**Before trusting these numbers, two actions:**
1. **Verify** the 2017–2021 figures against the 2021 report PDF — especially the cash-flow signs (bracket notation is easy to misread) and the 2020 column, which uses the *restated* figures (a "correction of errors" adjustment on p.83).
2. **Fill** 2022–2025 from each later report's equivalent "Consolidated figures" table, and back-fill `volume_mt` for 2017–2020 if the figure appears.

The `NaN`s are expected at this stage, not errors — the validation and completeness cells below are built to work with partial data, so the pipeline runs end-to-end now and gets richer as the data fills in.